# 02 Bronze: from daily landing files to one table per vendor

**Goal:** collect the 472 daily vendor files in `data/landing` (134 FusionSolar, 338 Solis) into one Bronze table per vendor, exactly as delivered: everything as text, nothing fixed, plus a `source_file` column.

Before loading, the files are checked: are any days missing, and do all files have the same columns? The landing files themselves are never edited.

## 1. Setup

In [ ]:
# Libraries: pathlib for file paths, pandas for tables
from pathlib import Path
import pandas as pd

# Project root: works whether the notebook runs from SOLAR or from SOLAR/notebooks
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
LANDING = ROOT / "data" / "landing"

# All daily files of each vendor, in date order (the * wildcard matches any name)
fs_files = sorted((LANDING / "fusionsolar").glob("*.csv"))
so_files = sorted((LANDING / "solis").glob("*.csv"))
print(len(fs_files), len(so_files))

134 338


## 2. Are any days missing?

Every file name carries its date (`fusionsolar_2024-12-31.csv`), so the date is a unique label for each day (a *natural key*). The check:

1. Take the date out of every file name.
2. Build the full calendar from the first to the last date.
3. Any calendar day without a file is a missing day.

In [2]:
# Take one file name apart to get its date
f = fs_files[0]
print(f.name)                  # file name with its ending
print(f.stem)                  # file name without .csv
print(f.stem.split("_"))       # the pieces between the underscores
print(f.stem.split("_")[1])    # the second piece is the date

fusionsolar_2024-12-31.csv
fusionsolar_2024-12-31
['fusionsolar', '2024-12-31']
2024-12-31


In [3]:
# FusionSolar: the date of every file
fs_days = []
for f in fs_files:
    day = f.stem.split("_")[1]
    fs_days.append(day)

# first day, last day, number of files
print(fs_days[0], fs_days[-1], len(fs_days))

2024-12-31 2025-06-01 134


In [4]:
# Solis: the date of every file
so_days = []
for f in so_files:
    day = f.stem.split("_")[1]
    so_days.append(day)

# first day, last day, number of files
print(so_days[0], so_days[-1], len(so_days))

2024-06-27 2025-06-01 338


In [5]:
# FusionSolar: every calendar day from the first to the last file, written like the file names
calendar = pd.date_range(fs_days[0], fs_days[-1]).strftime("%Y-%m-%d")

# A calendar day without a file is a missing day
fs_missing = []
for day in calendar:
    if day not in fs_days:
        fs_missing.append(day)

print(len(calendar), "days in the span")
print(len(fs_missing), "missing:", fs_missing)
print("Percentage of missing days:", len(fs_missing) / len(calendar) * 100)

153 days in the span
19 missing: ['2025-03-17', '2025-03-18', '2025-03-19', '2025-03-20', '2025-03-21', '2025-03-22', '2025-03-23', '2025-03-24', '2025-03-25', '2025-03-26', '2025-03-27', '2025-03-28', '2025-03-29', '2025-03-30', '2025-04-22', '2025-04-23', '2025-04-24', '2025-04-25', '2025-04-26']
Percentage of missing days: 12.418300653594772


In [6]:
# Solis: the same check
calendar1 = pd.date_range(so_days[0], so_days[-1]).strftime("%Y-%m-%d")

so_missing = []
for day in calendar1:
    if day not in so_days:
        so_missing.append(day)

print(len(calendar1), "days in the span")
print(len(so_missing), "missing:", so_missing)
print("Percentage of missing days:", (len(so_missing) / len(calendar1)) * 100)

340 days in the span
2 missing: ['2024-07-20', '2024-07-21']
Percentage of missing days: 0.5882352941176471


**Result**

| Vendor | First day | Last day | Days in span | Files | Missing days | Missing |
| --- | --- | --- | --- | --- | --- | --- |
| FusionSolar | 2024-12-31 | 2025-06-01 | 153 | 134 | 19 | 12.4 % |
| Solis | 2024-06-27 | 2025-06-01 | 340 | 338 | 2 | 0.6 % |

- FusionSolar is missing two blocks: 17 to 30 March 2025 (14 days) and 22 to 26 April 2025 (5 days).
- Solis is missing 20 and 21 July 2024.
- The time spans differ because the vendors monitor different plants: Solis has 1 plant, monitored since June 2024, and FusionSolar has 6 plants, monitored since January 2025.

In [7]:
# The missing-days check as a table, one row per vendor
summary = pd.DataFrame({
    "vendor": ["fusionsolar", "solis"],
    "first_day": [fs_days[0], so_days[0]],
    "last_day": [fs_days[-1], so_days[-1]],
    "files": [len(fs_days), len(so_days)],
    "missing_days": [len(fs_missing), len(so_missing)],
    "missing_percentage": [len(fs_missing) / len(calendar) * 100, len(so_missing) / len(calendar1) * 100]
})

# Save it in reports/ (outside data/, so git keeps it)
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)   # create the folder if it does not exist yet
summary.to_csv(REPORTS / "landing_days_check.csv", index=False)
summary

,vendor,first_day,last_day,files,missing_days,missing_percentage
0,fusionsolar,2024-12-31,2025-06-01,134,19,12.418301
1,solis,2024-06-27,2025-06-01,338,2,0.588235


## 3. Do all files have the same columns?

The **schema** (the column names and how many there are) must be the same in every file of a vendor. Only the number of rows may change from day to day.

Expected: **FusionSolar 9 columns**, **Solis 8 columns** (Solis has no `efficiency`).

Files are read with `dtype=str`: everything as text, so pandas guesses nothing.

In [8]:
# Read one file as text and return its shape: (rows, columns)
def file_shape(f):
    df = pd.read_csv(f, dtype=str)
    return df.shape

In [9]:
# FusionSolar: rows and columns of every file
for f in fs_files:
    rows, cols = file_shape(f)
    print(f.name, rows, cols)

fusionsolar_2024-12-31.csv 24 9
fusionsolar_2025-01-01.csv 4622 9
fusionsolar_2025-01-02.csv 5466 9
fusionsolar_2025-01-03.csv 5467 9
fusionsolar_2025-01-04.csv 3484 9
fusionsolar_2025-01-05.csv 3593 9
fusionsolar_2025-01-06.csv 4225 9
fusionsolar_2025-01-07.csv 4653 9
fusionsolar_2025-01-08.csv 5520 9
fusionsolar_2025-01-09.csv 5437 9
fusionsolar_2025-01-10.csv 5515 9
fusionsolar_2025-01-11.csv 4680 9
fusionsolar_2025-01-12.csv 4775 9
fusionsolar_2025-01-13.csv 3891 9
fusionsolar_2025-01-14.csv 3998 9
fusionsolar_2025-01-15.csv 4159 9
fusionsolar_2025-01-16.csv 4543 9
fusionsolar_2025-01-17.csv 4490 9
fusionsolar_2025-01-18.csv 4310 9
fusionsolar_2025-01-19.csv 4777 9
fusionsolar_2025-01-20.csv 4493 9
fusionsolar_2025-01-21.csv 4422 9
fusionsolar_2025-01-22.csv 4004 9
fusionsolar_2025-01-23.csv 3992 9
fusionsolar_2025-01-24.csv 3535 9
fusionsolar_2025-01-25.csv 4906 9
fusionsolar_2025-01-26.csv 4922 9
fusionsolar_2025-01-27.csv 4963 9
fusionsolar_2025-01-28.csv 4950 9
fusionsolar_2025

In [10]:
# Solis: print only the files that do not have the expected 8 columns
for f in so_files:
    rows, cols = file_shape(f)
    if cols != 8:
        print(f.name, rows, cols)

solis_2024-07-02.csv 1090 1
solis_2024-07-29.csv 950 1
solis_2024-09-05.csv 1016 1
solis_2024-09-13.csv 1004 1
solis_2024-10-04.csv 926 1
solis_2024-10-14.csv 928 1
solis_2024-10-26.csv 1044 1
solis_2024-11-10.csv 1474 1
solis_2024-11-19.csv 1453 1
solis_2024-12-26.csv 1640 1
solis_2025-01-17.csv 1637 1
solis_2025-01-26.csv 1914 1
solis_2025-02-02.csv 1935 1
solis_2025-02-16.csv 2021 1
solis_2025-04-03.csv 1200 1
solis_2025-04-09.csv 2115 1
solis_2025-04-30.csv 2271 1


**Result:** 17 Solis files are read as **1 column**. They use `;` as the separator and `,` as the decimal mark (`32,6`), the European export format. Read with the default `,`, their whole header becomes one column name.

If these files were stacked with the others, the big table would get 9 columns instead of 8, with no error message: a *silent failure*.

In [11]:
# Smallest and largest number of rows and columns over all files of one vendor
def shape_summary(vendor, files):
    all_rows = []
    all_cols = []
    for f in files:
        rows, cols = file_shape(f)
        all_rows.append(rows)
        all_cols.append(cols)
    return {
        "vendor": vendor,
        "min_rows": min(all_rows),
        "max_rows": max(all_rows),
        "min_cols": min(all_cols),
        "max_cols": max(all_cols),
    }

# One row per vendor (each dictionary becomes one row)
shape_report = pd.DataFrame([
    shape_summary("fusionsolar", fs_files),
    shape_summary("solis", so_files),
])
shape_report

,vendor,min_rows,max_rows,min_cols,max_cols
0,fusionsolar,1,8005,9,9
1,solis,11,2955,1,8


In [12]:
# Save the schema check next to the missing-days report
shape_report.to_csv(REPORTS / "landing_shape_check.csv", index=False)

**Result**

| Vendor | Min rows | Max rows | Min columns | Max columns |
| --- | --- | --- | --- | --- |
| FusionSolar | 1 | 8005 | 9 | 9 |
| Solis | 11 | 2955 | 1 | 8 |

- FusionSolar: every file has 9 columns. Schema OK.
- Solis: between 1 and 8 columns. The 17 `;` files break the schema.
- One FusionSolar file has only 1 row: `fusionsolar_2025-06-01.csv`, the last day. Rows may vary, but it is worth remembering.

## 4. Read every file with its own separator

The landing files are the record of what the vendor sent, so they are **never edited**. Replacing `;` with `,` would also break `32,6` into two values. Instead, each file is read with its own separator:

1. Look at the header line. It holds only column names, so a decimal comma cannot confuse it.
2. If it contains `;`, read the file with `;`, otherwise with `,`.
3. Read everything as text (`dtype=str`). The decimal commas stay as they are; Silver converts them later.
4. Add a `source_file` column (**lineage**). After stacking, every row still knows which file it came from. Silver needs this for duplicates, for ambiguous dates and for tracing problems back.

In [13]:
# Read one landing file with the right separator, as text, and label its rows with the file name
def read_landing_file(f):
    header = f.read_text().splitlines()[0]   # first line = the column names
    if ";" in header:
        sep = ";"
    else:
        sep = ","
    df = pd.read_csv(f, sep=sep, dtype=str, keep_default_na=False)
    df["source_file"] = f.name               # lineage: the file this row came from
    return df

**Result:** all 338 Solis files now read as 9 columns. For example, `solis_2024-07-02.csv` went from `(1090, 1)` to `(1090, 9)`: the same rows, now in the right columns.


In [14]:
# Solis Bronze: all files stacked into one table
tables = []
for f in so_files:
    tables.append(read_landing_file(f))

bronze_solis = pd.concat(tables, ignore_index=True)
print(bronze_solis.shape)

(509342, 9)


In [16]:
# FusionSolar Bronze: all files stacked into one table
tables = []
for f in fs_files:
    tables.append(read_landing_file(f))

bronze_fs = pd.concat(tables, ignore_index=True)
print(bronze_fs.shape)

(725604, 10)


In [17]:
# Bronze folder, next to landing
BRONZE = ROOT / "data" / "bronze"
BRONZE.mkdir(exist_ok=True)

bronze_solis.to_parquet(BRONZE / "solis.parquet", index=False)
bronze_fs.to_parquet(BRONZE / "fusionsolar.parquet", index=False)

In [18]:
check = pd.read_parquet(BRONZE / "solis.parquet")
print(check.shape)

(509342, 9)


In [19]:
# Reference tables (devices, stations, ...) to Bronze, one Parquet file each
for f in sorted((LANDING / "reference").glob("*.csv")):
    df = read_landing_file(f)
    df.to_parquet(BRONZE / f"{f.stem}.parquet", index=False)
    print(f.stem, df.shape)

devices (66, 8)
inverter_state_canonical (3, 3)
inverter_state_mapping_fusionsolar (26, 4)
inverter_state_mapping_solis (3, 4)
source (2, 3)
stations (7, 9)


In [20]:
# Schema report of the 6 reference tables
schema_rows = []
for f in sorted((LANDING / "reference").glob("*.csv")):
    df = read_landing_file(f)
    schema_rows.append({
        "table": f.stem,
        "rows": df.shape[0],
        "columns": df.shape[1] - 1,                  # minus our source_file
        "column_names": ", ".join(df.columns[:-1]),  # all names except source_file
    })

reference_schema = pd.DataFrame(schema_rows)
reference_schema.to_csv(REPORTS / "reference_schema.csv", index=False)
reference_schema

,table,rows,columns,column_names
0,devices,66,7,"device_hash_id, source, station_hash_id, devic..."
1,inverter_state_canonical,3,2,"canonical_state_id, description"
2,inverter_state_mapping_fusionsolar,26,3,"state_id, description, canonical_state_id"
3,inverter_state_mapping_solis,3,3,"state_id, description, canonical_state_id"
4,source,2,2,"id, name"
5,stations,7,8,"station_hash_id, source, station_id, name, loc..."
